<!-- nav -->
[← 8 · SystemVerilog Constraints](08_16_SystemVerilog_Constraints.ipynb) · [Home](../../README.md) · [9 · Constraints as data →](09_Constraints_As_Data.ipynb)

# 8.17 · SystemVerilog Preprocessor

Verilog's and SystemVerilog's compiler directives (IEEE 1800 clause 22) are written among the code they direct: a macro
is defined, `` `define WIDTH 8 ``, and used where an expression stands, `` `WIDTH - 1 ``; a conditional directive keeps
code or drops it, `` `ifdef SIM ... `endif ``. Tutorial 8 showed a `` `define `` (08.3). This page builds the dialect
that models them, `SystemVerilog Preprocessor`, from a macro usage to the conditional directives, each in the builders'
fluent calls beside the text SystemVerilog writes, the comments kept among them too. Then what is done with them: held
among an expression's terms and a constraint block's items, validated, with one refused; never expanded or evaluated,
each kind refused by name; and the editions that have them, every one of Verilog's and SystemVerilog's.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `name` and
`literal` write a name and a literal of SystemVerilog's `Expressions` (tutorials 3 and 4), and `spelled` an identifier,
a macro's name; `refused` gives what an evaluator says of an expression, its value or the reason it refuses it, as
tutorial 8's does.

In [1]:
import assert from "node:assert/strict";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVPreprocessor from "@mbse/expressions/SystemVerilog/Preprocessor";
import { Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const Preprocessor = SVPreprocessor.Expressions.DIALECT; // the dialect, as the framework reads it
const name = (spelling: string) => (d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling(spelling))));
const literal = (value: unknown) => (d: any) => d.literal((l: any) => l.value((v: any) => v.as_scalar(value)));
const spelled = (spelling: string) => (i: any) => i.spelling(spelling); // an identifier, of its spelling
const x = name("x");


/** What `evaluate` says of `expression`: its value, or the reason it refuses it. */
function refused(evaluate: (expression: any) => unknown, expression: any): string {
  try {
    return String(evaluate(expression));
  } catch (error) {
    return (error as Error).message;
  }
}

<a id="T08.17.1"></a>
## 08.17.1 · A macro usage, where an expression stands

```text
`WIDTH        `MAX(a, b)        `WIDTH - 1
```

A macro usage is a `macrousage`: its name, an identifier, and its arguments, kept as the text they are written in, since
a macro's arguments are text the preprocessor substitutes, not terms. It stands wherever an expression may: every one of
the family's `Term`s lists it, `Expressions`' among them, so `` `WIDTH - 1 `` is `Expressions`' own binary expression,
its left operand a macro usage:

In [2]:
const usages: any[] = store.transaction(() => [
  (u: any) => u.macrousage((m: any) => m.name(spelled("WIDTH"))),
  (u: any) => u.macrousage((m: any) => m.name(spelled("MAX")).arguments("a, b")),
].map((root) => S.SystemVerilog.Preprocessor.Expression((e: any) => e.root(root))));
const less = store.transaction(() => S.SystemVerilog.Expressions.Expression((e: any) => e.root((d: any) => d.binaryexpression((b: any) => b.left((d: any) => d.macrousage((m: any) => m.name(spelled("WIDTH")))).operator("-").right(literal(1n))))));
for (const expression of [...usages, less]) {
  const form = Terms.form(Terms.term(expression));
  console.log(SystemVerilog.Text.ToText(expression).padEnd(11), form.kind.padEnd(16),
    form.arguments.map((t: any) => `${Terms.kindOf(t)[0]} of ${Preprocessor.owner(t).name()}`).join(", ") || "-");
}
assert.deepEqual([...usages, less].map((e) => SystemVerilog.Text.ToText(e)), ["`WIDTH", "`MAX(a, b)", "`WIDTH - 1"]);

`WIDTH      macrousage       -


`MAX(a, b)  macrousage       -


`WIDTH - 1  binaryexpression macrousage of SystemVerilog Preprocessor, literal of SystemVerilog


<a id="T08.17.2"></a>
## 08.17.2 · Directives

```text
`define WIDTH 8        `define MAX(a, b) ((a) > (b) ? (a) : (b))        `undef WIDTH
`include "defs.svh"    `timescale 1ns / 1ps    `default_nettype none    `resetall
```

Each directive is a kind of its own: `definedirective` (its name, whether it is function-like, its parameters, and its
body, kept as the text it is, since a macro's body is text until it is substituted), `undefdirective`,
`includedirective`, `timescaledirective` and `defaultnettypedirective`. A directive the dialect has no kind for is an
`otherdirective`, kept as its text:

In [3]:
const directives: any[] = store.transaction(() => [
  (u: any) => u.definedirective((d: any) => d.name(spelled("WIDTH")).body("8")),
  (u: any) => u.definedirective((d: any) => d.name(spelled("MAX")).function_like(true)
    .parameters([spelled("a"), spelled("b")]).body("((a) > (b) ? (a) : (b))")),
  (u: any) => u.undefdirective((d: any) => d.name(spelled("WIDTH"))),
  (u: any) => u.includedirective((d: any) => d.path("defs.svh")),
  (u: any) => u.timescaledirective((d: any) => d.unit("1ns").precision("1ps")),
  (u: any) => u.defaultnettypedirective((d: any) => d.net_type("none")),
  (u: any) => u.otherdirective((d: any) => d.text("`resetall")),
].map((root) => S.SystemVerilog.Preprocessor.Expression((e: any) => e.root(root))));
for (const directive of directives) {
  console.log(SystemVerilog.Text.ToText(directive).padEnd(43), Terms.kindOf(Terms.term(directive))[0]);
}
assert.deepEqual(directives.map((d) => SystemVerilog.Text.ToText(d)), [
  "`define WIDTH 8", "`define MAX(a, b) ((a) > (b) ? (a) : (b))", "`undef WIDTH", '`include "defs.svh"',
  "`timescale 1ns / 1ps", "`default_nettype none", "`resetall"]);

`define WIDTH 8                             definedirective


`define MAX(a, b) ((a) > (b) ? (a) : (b))   definedirective


`undef WIDTH                                undefdirective


`include "defs.svh"                         includedirective


`timescale 1ns / 1ps                        timescaledirective


`default_nettype none                       defaultnettypedirective


`resetall                                   otherdirective


<a id="T08.17.3"></a>
## 08.17.3 · Conditional directives

```text
`ifdef SIM            `ifdef (FAST && !SIM)
x < 4;                x < 16;
`else                 `elsif SIM
x < 8;                x < 4;
`endif                `endif
```

A conditional directive, `ifdefdirective`, keeps its items where its macro is defined (`` `ifndef ``, `negated`, where
it is not), its `` `elsif `` branches, `ElsifDirective` helpers, where theirs is, and its alternative, after `` `else ``,
where none is. IEEE 1800-2023 lets it test a condition of macros' names, in parentheses, in place of one name. Its items
are what it keeps: here constraints, the `Term` of `SystemVerilog Constraints`, which lists the directives in turn,
so that a constraint block may hold them. The printer writes each directive at the start of a line of its own:

In [4]:
const simple_ifdef = store.transaction(() => S.SystemVerilog.Preprocessor.Expression((e: any) => e.root((u: any) => u.ifdefdirective((d: any) => d.name(spelled("SIM"))
  .items([(c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(x).operator("<").right(literal(4n)))))]).alternative([(c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(x).operator("<").right(literal(8n)))))]).has_else(true)))));
const condition = (d: any) => d.parenthesizedexpression((p: any) => p.expression((d: any) => d.binaryexpression((b: any) => b.left(name("FAST")).operator("&&").right((n: any) => n.unaryexpression((u: any) => u.operator("!").operand(name("SIM")))))));
const chosen = store.transaction(() => S.SystemVerilog.Preprocessor.Expression((e: any) => e.root((u: any) => u.ifdefdirective((d: any) => d.condition(condition)
  .items([(c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(x).operator("<").right(literal(16n)))))])
  .branches([(b: any) => b.name(spelled("SIM")).items([(c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(x).operator("<").right(literal(4n)))))])])))));
for (const directive of [simple_ifdef, chosen]) {
  console.log(SystemVerilog.Text.ToText(directive));
  console.log("--", "free:", [...Symbolics.free(directive, Preprocessor)].sort().join(", "));
}
assert.equal(SystemVerilog.Text.ToText(chosen), "`ifdef (FAST && !SIM)\nx < 16;\n`elsif SIM\nx < 4;\n`endif");

`ifdef SIM
x < 4;
`else
x < 8;
`endif


-- free: x


`ifdef (FAST && !SIM)
x < 16;
`elsif SIM
x < 4;
`endif


-- free: x


`FAST` and `SIM` are macros' names, not values: a conditional directive binds them within its condition alone, so
`Symbolics.free` gives only `x`, the name its items read, which the scope must bind as anywhere else. Where a reader
keeps the text a conditional drops without reading it, that text is a `disabledtext`, as written.

<a id="T08.17.4"></a>
## 08.17.4 · Among constraints, with comments

```text
p.randomize() with { // at least one
x > 0;
`ifdef SIM
x < 4;
`endif
}
```

Since the Constraints dialect's `Term` lists the directives and the dialect's `comment`, `randomize() with` holds them
among its constraints, as mbse-programs reads them from source. The printer lays them out as a program is laid out, a
line comment ending its line and a directive on lines of its own:

In [5]:
const randomized = store.transaction(() => S.SystemVerilog.Expressions.Expression((e: any) => e.root((u: any) => u.randomizewithexpression((r: any) => r
  .call((c: any) => c.callexpression((m: any) => m.callee((d: any) => d.memberexpression((b: any) => b
    .value(name("p")).member(spelled("randomize")))).arguments([])))
  .items([(c: any) => c.comment((t: any) => t.text(" at least one")), (c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(x).operator(">").right(literal(0n))))),
    (c: any) => c.ifdefdirective((d: any) => d.name(spelled("SIM")).items([(c: any) => c.expressionconstraint((e: any) => e.expression((d: any) => d.binaryexpression((b: any) => b.left(x).operator("<").right(literal(4n)))))]))])))));
console.log(SystemVerilog.Text.ToText(randomized));
const items = Terms.form(Terms.term(randomized)).arguments.slice(1);
console.log("--", items.map((t: any) => `${Terms.kindOf(t)[0]} of ${Preprocessor.owner(t).name()}`).join(", "));
assert.equal(SystemVerilog.Text.ToText(randomized), "p.randomize() with { // at least one\nx > 0;\n`ifdef SIM\nx < 4;\n`endif\n}");

p.randomize() with { // at least one
x > 0;
`ifdef SIM
x < 4;
`endif
}


-- comment of SystemVerilog Preprocessor, expressionconstraint of SystemVerilog Constraints, ifdefdirective of SystemVerilog Preprocessor


<a id="T08.17.5"></a>
## 08.17.5 · Validated, and one refused

`validate` checks each kind's own rules, with mbse-programs' messages: a directive's name an identifier, a default net
type one IEEE 1800 names, a conditional directive's name or its condition, not both. A condition in place of a name is
written in parentheses, so one without them is refused:

In [6]:
const checked: any[] = [simple_ifdef, ...store.transaction(() => [
  (u: any) => u.defaultnettypedirective((d: any) => d.net_type("wirex")),
  (u: any) => u.ifdefdirective((d: any) => d
    .condition((d: any) => d.binaryexpression((b: any) => b.left(name("FAST")).operator("&&").right(name("SIM"))))
    .items([(c: any) => c.expressionconstraint((e: any) => e.expression(
      (d: any) => d.binaryexpression((b: any) => b.left(x).operator("<").right(literal(16n)))))])),
].map((root) => S.SystemVerilog.Preprocessor.Expression((e: any) => e.root(root)))) as any[]];
for (const directive of checked) {
  console.log(SystemVerilog.Text.ToText(directive).split("\n")[0]!.padEnd(24), "|", Preprocessor.validate(directive, ["x"]).join("; ") || "valid");
}
assert.deepEqual(Preprocessor.validate(checked[2], ["x"]), ["an IfdefDirective's condition is in parentheses"]);

`ifdef SIM               | valid


`default_nettype wirex   | DefaultNettypeDirective.net_type cannot be 'wirex'


`ifdef FAST && SIM       | an IfdefDirective's condition is in parentheses


<a id="T08.17.6"></a>
## 08.17.6 · Never expanded, never evaluated

No macro is expanded and no directive is run: the dialect models the text a preprocessor reads, not what it makes of
it. SystemVerilog's evaluator refuses each kind by name (stage 6, "Q11"), held alone or as an expression's operand:
`` `WIDTH - 1 `` has no value until a preprocessor gives `` `WIDTH `` one:

In [7]:
const evaluate = SystemVerilog.Evaluators.OfAny;
for (const expression of [usages[0], directives[0], simple_ifdef, less]) {
  console.log(SystemVerilog.Text.ToText(expression).split("\n")[0]!.padEnd(16), refused((e) => evaluate(e, { x: 1n }), expression));
}

`WIDTH           SystemVerilog Preprocessor macrousage is not evaluated


`define WIDTH 8  SystemVerilog Preprocessor definedirective is not evaluated


`ifdef SIM       SystemVerilog Preprocessor ifdefdirective is not evaluated


`WIDTH - 1       SystemVerilog Preprocessor macrousage is not evaluated


<a id="T08.17.7"></a>
## 08.17.7 · Its editions are data

Compiler directives are Verilog's as much as SystemVerilog's, so the dialect covers all eight editions, IEEE Std
1364-1995 to 1800-2023, each with every one of its kinds
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):

In [8]:
const singleton = store.Instances.Expressions.SystemVerilog.Preprocessor;
const kinds = new Map((store.registered(Preprocessor.union).branches as any[]).map((branch) => [branch.schema, branch.name]));
for (const [designation, edition] of Standards.standards(singleton)) {
  console.log(designation.padEnd(19), Standards.tags(edition)[1]!.padEnd(19), Standards.kinds(edition).length, "kinds");
}
console.log("each:", Standards.kinds(Standards.find(singleton, "Verilog-1995")!).map((k) => kinds.get(k)).join(", "));

IEEE Std 1364-1995  Verilog-1995        10 kinds


IEEE Std 1364-2001  Verilog-2001        10 kinds


IEEE Std 1364-2005  Verilog-2005        10 kinds


IEEE Std 1800-2005  SystemVerilog-2005  10 kinds


IEEE Std 1800-2009  SystemVerilog-2009  10 kinds


IEEE Std 1800-2012  SystemVerilog-2012  10 kinds


IEEE Std 1800-2017  SystemVerilog-2017  10 kinds


IEEE Std 1800-2023  SystemVerilog-2023  10 kinds


each: macrousage, includedirective, definedirective, undefdirective, timescaledirective, defaultnettypedirective, ifdefdirective, disabledtext, otherdirective, comment


An edition records the kinds it has, not each of a kind's forms: a conditional directive's condition in parentheses is
IEEE 1800-2023's, but `ifdefdirective` is in every edition's.

## What you learned

- **Directives, macro usages and comments are terms of `SystemVerilog Preprocessor`**, each in the builders' calls
  beside its text; a macro's body and arguments are text, kept as written.
- **A macro usage stands wherever an expression does**, and the directives and comments among a constraint block's
  items, which a conditional directive's items are in turn.
- **A conditional directive's condition names macros**, bound within it alone; it is validated with mbse-programs'
  messages, a condition without parentheses refused.
- **Nothing is expanded or evaluated**: each kind is refused by name; its editions are all of Verilog's and
  SystemVerilog's.

That ends SystemVerilog's sublanguages. Next, Part II puts expressions to work as constraints, starting with case
study 9.

<!-- nav -->
[← 8 · SystemVerilog Constraints](08_16_SystemVerilog_Constraints.ipynb) · [Home](../../README.md) · [9 · Constraints as data →](09_Constraints_As_Data.ipynb)